# Leakage-safe user feature extraction

Builds **user** features from the purchase log (here the review/comment data
in `data/Home_and_Kitchen_filtered.csv`). The output has **one row per
purchase event**, and every feature is computed only from that user's
purchases on **strictly earlier dates** — no leakage from the current
purchase or anything on the same day / later.

If a user bought on Mar 1, Apr 2, May 3:
- the **Mar 1** row has empty history (all features NaN),
- the **Apr 2** row sees only Mar 1,
- the **May 3** row sees Mar 1 and Apr 2.

Category/brand fields (`cat_2/3/4`, `brand`) are not in the purchase log, so
they are joined in from `data/df_features.pkl` on `asin`.

This notebook is the runnable companion to
`feature_extraction_workflow/extract_features_user.py`.

In [ ]:
import sys
from pathlib import Path

# Make the package importable when running from this folder
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from feature_extraction_workflow import (
    load_purchases,
    compute_user_features,
    run_user_feature_extraction,
    save_user_features,
)
from feature_extraction_workflow.extract_features_user import FEATURE_COLUMNS

pd.options.display.max_columns = None
pd.options.display.width = None

DATA_DIR = PROJECT_ROOT / 'data'
REVIEWS_PATH = DATA_DIR / 'Home_and_Kitchen_filtered.csv'
FEATURES_PATH = DATA_DIR / 'df_features.pkl'
OUT_PATH = DATA_DIR / 'df_user_features.pkl'

## 1. Load the purchase log and attach item category/brand

`asin` is read as a **string** so leading zeros survive (otherwise the join
to `df_features` silently misses). Rows without a usable `unixReviewTime` are
dropped — they can't be placed on the timeline.

In [ ]:
df = load_purchases(REVIEWS_PATH, FEATURES_PATH)
print('purchases:', df.shape)
print('unique users:', df['reviewerID'].nunique())
print('rows with no item match (NaN brand):', df['brand'].isna().sum())
df.head(3)

## 2. Sanity-check the leakage logic on a tiny synthetic example

The Mar/Apr/May case plus a user with two same-day purchases. Expect: the
first purchase (and same-day purchases) have empty history; same-day
purchases become history only for *later* dates; the inter-purchase gap
between two same-day purchases is 0.

In [ ]:
def _ts(y, m, d):
    return int(pd.Timestamp(year=y, month=m, day=d, tz='UTC').timestamp())

demo = pd.DataFrame(
    [
        ('U1', 'aaa', _ts(2021, 3, 1), 'Home & Kitchen', 'Bakeware', 'Cake Pans', 'Wilton'),
        ('U1', 'bbb', _ts(2021, 4, 2), 'Home & Kitchen', 'Bakeware', 'Cake Pans', 'Wilton'),
        ('U1', 'ccc', _ts(2021, 5, 3), 'Kitchen & Dining', 'Glassware', 'Mugs', 'Acme'),
        ('U2', 'ddd', _ts(2021, 1, 10), 'Home & Kitchen', 'Bakeware', 'Cake Pans', 'Wilton'),
        ('U2', 'eee', _ts(2021, 1, 10), 'Home & Kitchen', 'Bakeware', 'Cake Pans', None),
        ('U2', 'fff', _ts(2021, 2, 15), 'Bath', 'Towels', 'Bath Towels', 'Acme'),
    ],
    columns=['reviewerID', 'asin', 'unixReviewTime', 'cat_2', 'cat_3', 'cat_4', 'brand'],
)
demo_feats = compute_user_features(demo)
pd.concat([demo[['reviewerID', 'asin']], demo_feats[[
    'prior_purchase_count', 'distinct_items', 'distinct_brands',
    'avg_inter_purchase_gap', 'recency', 'days_since_first_purchase',
    'brand_loyalty', 'favorite_cat_3', 'preferred_season']]], axis=1)

## 3. Compute the leakage-safe user features on the full log

One row per purchase event. For each row, features reflect only the user's
strictly-earlier-date purchases. This is a single ordered pass over the data;
it takes a few minutes on the full ~6.9M rows.

In [ ]:
user_feats = compute_user_features(df)
print('feature matrix:', user_feats.shape)
user_feats[FEATURE_COLUMNS].describe(include='all').T

In [ ]:
# Assemble keys + features (one row per purchase event)
keys = ['reviewerID', 'asin', 'reviewTime', 'unixReviewTime']
df_user = pd.concat([df[keys].reset_index(drop=True),
                     user_feats.reset_index(drop=True)], axis=1)

n_cold = df_user['prior_purchase_count'].isna().sum()
print('cold-start rows (first purchase, empty history):',
      f'{n_cold:,} / {len(df_user):,}')
df_user.head(5)

### Spot-check one user's timeline

Pick an active user and confirm features grow monotonically with their
history (counts non-decreasing, recency resets at each purchase).

In [ ]:
top_user = df_user['reviewerID'].value_counts().index[0]
cols = ['reviewTime', 'prior_purchase_count', 'distinct_items',
        'recency', 'avg_inter_purchase_gap', 'favorite_cat_3', 'favorite_brand']
(df_user[df_user['reviewerID'] == top_user]
 .sort_values('unixReviewTime')[cols]
 .head(15))

## 4. Save to pickle

Pickle so the (few) object columns round-trip cleanly. This is the file the
two-tower notebook can join to the interactions on `(reviewerID, asin, time)`.

In [ ]:
save_user_features(df_user, OUT_PATH)
print(f'saved {len(df_user):,} rows x {df_user.shape[1]} cols -> {OUT_PATH}')

Equivalently, the whole pipeline in one call:

```python
df_user = run_user_feature_extraction(REVIEWS_PATH, FEATURES_PATH)
save_user_features(df_user, OUT_PATH)
```